In [ ]:
# Step 1:
#! pip install datasets huggingface_hub pandas pillow
#! pip install -U huggingface_hub

from huggingface_hub import snapshot_download
import os
import pandas as pd
from pathlib import Path
import tarfile

# step 2: check dataset:
DATASET_REPO = "RafeiKAr/dataset_gaze"

LOCAL_DATASET_DIR = "./data"

# step 2.1: downlaod if need:
if not os.path.exists(LOCAL_DATASET_DIR):
    print("Download Dataset from Huggingface.co: ...")
    eingabe = input("are you sure to downlaod the 20 GB-Dataset?")
    if eingabe == "yes":
        snapshot_download(
            repo_id=DATASET_REPO,
            repo_type="dataset",
            local_dir=LOCAL_DATASET_DIR,
            local_dir_use_symlinks=False
        )

        print("Download fertig.")
        
        # step 2.2: extract tar-file:
        out = Path("images")
        out.mkdir(exist_ok=True)

        for tar_path in sorted(Path(".").glob("images_*.tar")):
            print("Extracting", tar_path)

            with tarfile.open(tar_path) as tar:
                tar.extractall(out)

        print("Done")

    else:
        pass

else:
    print("Dataset exist already!!.")

In [ ]:
# step 3:

images_dir = "./data/images"
labels_file = "./data/labels.csv"

#images_dir  = "F:/Datasets/processed_dataset/images"
#labels_file = "F:/Datasets/processed_dataset/labels.csv"


#df = pd.read_csv("./data/labels.csv")
df = pd.read_csv(labels_file)

print(df.head())
#print("Anzahl Samples:", len(df))


# Step 4:
print("Anzahl Samples:", len(df))


In [ ]:
# Step 5:

import matplotlib.pyplot as plt

x = df["x"]
y = df["y"]

plt.figure(figsize=(6,6))
plt.scatter(x, y, s=2)

plt.title("Gaze Targets")
plt.xlabel("X")
plt.ylabel("Y")

plt.gca().invert_yaxis()
plt.show()


screen_w = df["x"].max()
screen_h = df["y"].max()
print(screen_w, screen_h)


In [ ]:
# Step 6.1. Constante Baseline

import numpy as np

x_true = df["x"].values
y_true = df["y"].values

x_const = np.mean(x_true)
y_const = np.mean(y_true)

pred_x_const = np.full_like(x_true, x_const)
pred_y_const = np.full_like(y_true, y_const)

errors = np.sqrt(
    (x_true - pred_x_const)**2 +
    (y_true - pred_y_const)**2
)

print("\n\nKonstante Baseline:")
print(f"Mean Error: {np.mean(errors)}")
#print("Median Error:", np.median(errors))
print(f"Error %: {100*np.mean(errors)/screen_w:.2f}%")
print(f"Diagonal-Error %: {100*np.round(np.mean(errors) / np.sqrt(2*screen_w**2), 3)} %")

In [ ]:
# 6.2. random Baseline

pred_x_random = np.random.uniform(
    0,
    screen_w,
    len(x_true)
)

pred_y_random = np.random.uniform(
    0,
    screen_h,
    len(y_true)
)

errors_random = np.sqrt(
    (x_true - pred_x_random)**2 +
    (y_true - pred_y_random)**2
)

print("\n\nRandom Baseline:")
print(f"Mean Error: {np.mean(errors_random)}")
print(f"Error %: {np.round(np.mean(errors_random) / screen_w, 3)*100} %")
print(f"Diagonal-Error %: {100*np.round(np.mean(errors_random) / np.sqrt(2*screen_w**2), 3)} %")